In [2]:
import chromadb 
import os
import numpy as np
from langchain_core.documents import Document
import uuid

In [ ]:
class VectorStore:

    def __init__(self, collections_name: str='documents', persistent_directory: str='../../vector_store'):
        
        self.collections_name=collections_name
        self.client=None
        self.persistent_directory=persistent_directory
        self.collection=None
        self._initialize_store()


    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistent_directory)
            self.collection=self.client.get_or_create_collection(self.collections_name)
            print(f"vector store is intailized with collection {self.collections_name} ")
            print(f" store has {self.collection.count()} documents ")
        except Exception as e:
            print(f'vector store is not loaded {e}')
            raise
    
    def add_documents(self, documents:list[Document], embeddings:np.ndarray):
        
        if( len(documents) != len(embeddings)):
            raise ValueError("embeddings are not corresponding to the documents")
        
        ids=[]
        metadata_list=[]
        content_list=[]
        embedding_list=[]

        if len(documents) ==0 :
            raise ValueError(' documents are empty')
        
        if self.collection is None:
            raise ValueError("Collection not initialized")
        
        for i, (doc, embed) in enumerate(zip(documents, embeddings)):

            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"

            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            metadata_list.append(metadata)

            content_list.append(doc.page_content)
            embedding_list.append(embed.tolist())
        
        try:
            
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                documents=content_list,
                metadatas=metadata_list
            )
            print(f"successfully documents are added")
            print(f"total collection size is {self.collection.count()}")
        except Exception as e:
            print(f" there was error while adding documents to vector store {e}")
            raise

    def query(self, query_embedding:np.ndarray, top_k:int):
        if self.collection is None:
            raise ValueError("vector collection is not available")
        results=self.collection.query([query_embedding], n_results=top_k)
        return results

        

In [4]:
vector_store=VectorStore()

vector store is intailized with collection documents 
 store has 0 documents 
